# Tutoriel 5

[Télécharger le tutoriel](../05_tutoriel.zip)

# Des tailles cohérentes en 1D

Dans un modèle numérique, la difficulté n'est presque jamais l'équation : c'est de **garder des tailles compatibles** d'une ligne à l'autre. Ce tutoriel ne résout aucune équation — on travaille sur un vecteur quelconque, et on n'imprime que des tailles.

Trois opérations suffisent, et il n'y en a pas d'autres :

| Opération | Écriture | Effet sur la taille |
|---|---|---|
| **dérivation** | `v[1:] - v[:-1]` | on perd une cellule |
| **moyenne** | `0.5*(v[1:] + v[:-1])` | on perd une cellule |
| **troncation** | `v[1:-1]` | on en perd **deux** |

## 1. Les trois opérations

La **dérivation** et la **moyenne** donnent la même taille, `n-1`. Ce n'est pas un hasard : toutes deux combinent deux cellules voisines, et le résultat ne vit plus *sur* les points de la grille mais **entre** eux. C'est précisément ce qui permet de les multiplier l'une par l'autre, comme dans `D * (v[1:] - v[:-1])`.

La **troncation**, elle, ne combine rien : elle renonce simplement à la première et à la dernière cellule.

In [1]:
import numpy as np

v = np.arange(9)*1.0
print("v                      :", v.shape)
print("v[1:] - v[:-1]         :", (v[1:] - v[:-1]).shape, " derivation  -> une cellule en moins")
print("0.5*(v[1:] + v[:-1])   :", (0.5*(v[1:] + v[:-1])).shape, " moyenne     -> une cellule en moins")
print("v[1:-1]                :", v[1:-1].shape, " troncation  -> deux cellules en moins")

v                      : (9,)
v[1:] - v[:-1]         : (8,)  derivation  -> une cellule en moins
0.5*(v[1:] + v[:-1])   : (8,)  moyenne     -> une cellule en moins
v[1:-1]                : (7,)  troncation  -> deux cellules en moins


## 2. L'enchaînement d'un modèle

Dans tous les modèles du cours, ces opérations s'enchaînent toujours de la même façon : on dérive une fois pour obtenir un **flux**, une seconde fois pour obtenir un **taux de variation**, et on met à jour ce qui reste.

```
v                              |-----|-----|-----|-----|-----|-----|     n
qx   = (v[1:] - v[:-1])/dx        |-----|-----|-----|-----|-----|       n-1
dvdt = -(qx[1:] - qx[:-1])/dx        |-----|-----|-----|-----|          n-2
v[1:-1] += dt * dvdt                 |-----|-----|-----|-----|          n-2
```

La dernière ligne est celle qui pose problème : `dvdt` a la taille `n-2`, il faut donc l'ajouter à **exactement** `n-2` cellules de `v`, et aux bonnes. Trois écritures sont possibles, et une seule est correcte :

- `choix = 1` → `v[1:-1] += ...` : les `n-2` cellules **du milieu** ;
- `choix = 2` → `v += ...` : les `n` cellules, donc une taille incompatible .

Exécutez avec `choix = 1`, puis essayez `2`.

In [ ]:
choix = 1       # <<< CHANGEZ CETTE LIGNE :  1 ou 2

dx = 1.0
dt = 0.1

v = np.zeros(9)
v[4] = 1.0                            # un pic au centre, tout le reste a zero
v_ini = np.copy(v)

qx   = ( v[1:] - v[:-1] )/dx          # taille n-1
dvdt = -( qx[1:] - qx[:-1] )/dx       # taille n-2
print("tailles :  v", v.shape, "  qx", qx.shape, "  dvdt", dvdt.shape)

if choix == 1:
    v[1:-1] += dt * dvdt              # les n-2 cellules du milieu
elif choix == 2:
    v       += dt * dvdt              # n cellules contre n-2 : incompatible
else:
    v[:-2]  += dt * dvdt              # n-2 cellules, mais decalees

print("avant   :", v_ini)
print("apres   :", np.round(v, 2))

tailles :  v (9,)   qx (8,)   dvdt (7,)
avant   : [0. 0. 0. 0. 1. 0. 0. 0. 0.]
apres   : [ 0.   0.  -0.1  0.2  0.9  0.   0.   0.   0. ]


Les trois écritures se comportent très différemment.

**`choix = 1`** donne `[0, 0, 0, -0.1, 1.2, -0.1, 0, 0, 0]` : la modification est **centrée sur le pic**, et symétrique — ce que l'on attend d'un problème symétrique.

**`choix = 2`** s'arrête sur

```
ValueError: operands could not be broadcast together with shapes (9,) (7,)
```

C'est une **bonne nouvelle** : Python refuse de mélanger des tailles incompatibles, et le message donne directement les deux tailles en conflit. La plupart des erreurs de dimensions se signalent ainsi. 

> **Le réflexe à prendre :** une erreur de *broadcast* est un cadeau. Le vrai danger, ce sont les tailles qui tombent juste par hasard — d'où l'intérêt d'imprimer les `.shape` quand un résultat surprend, et de vérifier qu'un problème symétrique donne une solution symétrique.

## 3. Un seul code pour plusieurs questions

Un exercice pose souvent deux ou trois questions qui ne diffèrent que par quelques paramètres. Plutôt que de dupliquer tout le code, on définit une variable `Q` en tête et on ne branche que la portion concernée :

In [3]:
Q = 1       # <<< CHANGEZ CETTE LIGNE : numero de la question, 1 ou 2

if Q == 1:
    duree = 600.0        # duree pour la question 1
    titre = "Question 1 : regime long"
elif Q == 2:
    duree = 90.0         # duree pour la question 2
    titre = "Question 2 : regime court"

print(titre, "-> duree =", duree)

Question 1 : regime long -> duree = 600.0


Tout le reste du code — initialisation, boucle temporelle, figure — **reste identique**. Seule change la poignée de lignes qui dépend de la question, et le titre de la figure peut reprendre `titre` pour qu'on sache toujours ce que l'on regarde.

**Attention :** on écrit `Q` en majuscule, car `qx` désigne déjà un flux.

L'intérêt est double : votre rendu tient en un seul fichier, et une correction faite dans la boucle profite automatiquement à toutes les questions.